# RelCheck — Full Pipeline Test (LLaVA + RelTR + Type-Aware Verification)

End-to-end test of the complete RelCheck v3 pipeline on LLaVA-generated captions.

## Pipeline

| Step | What |
|------|------|
| **Cell 1** | Setup: constants, Together.ai helpers |
| **Cell 2** | Load COCO images screened for RelTR-friendly entity pairs |
| **Cell 3** | Generate LLaVA-1.5-7B captions, then unload model |
| **Cell 4** | Load GroundingDINO + RelTR (after LLaVA freed) |
| **Cell 5** | All helpers: KB construction · type-aware verify · KB-first cascade |
| **Cell 6** | **Full pipeline loop** (KB → RelTR → triple extraction → verify → correct) |
| **Cell 7** | R-POPE LLM-Judge evaluation |
| **Cell 8** | Coverage analysis + RelTR contribution breakdown |

## Key design contributions (paper-defensible)

1. **Type-aware verification** — SPATIAL triples use deterministic bbox geometry (GroundingDINO); ACTION/ATTRIBUTE use crop-based contrastive VQA. No prior work splits by relation type.
2. **RelTR as Stage 0 corrector** — visual classifier (not generative), so zero hallucination risk from the correction source itself.
3. **KB-first cascade** — 4-stage decreasing-risk cascade: RelTR → KB description → constrained VLM selection → open VLM.
4. **MEDIUM + HIGH confidence gate** — accepts borderline hallucinations (not just 3-to-0 NO votes).
5. **Coherency check** — LLM judge before applying any correction (prevents garbled captions).

## Run order
1. Set `TOGETHER_API_KEY` in Cell 1
2. Run COCO screening notebook first → produces `reltr_friendly_images.json`
3. Run Cells 1 → 8 top-to-bottom
4. Set `N_IMAGES=50` for validation; `N_IMAGES=100` for full run


In [ ]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────────
import os, sys, json, re, time, gc, random
from pathlib import Path
from PIL import Image
import torch
import requests

# ── Config ────────────────────────────────────────────────────────────────────
TOGETHER_API_KEY  = "YOUR_TOGETHER_API_KEY"   # ← paste your key
N_IMAGES          = 50                         # start small; scale to 100 later
CONF_THRESHOLD    = 0.3                        # RelTR confidence threshold

# Confidence thresholds (type-aware verification)
SUPPORTED_THRESHOLD  = 0.65   # yes_ratio >= → supported (not hallucinated)
UNCERTAIN_LOW        = 0.50   # yes_ratio in [0.50, 0.65) → uncertain, skip
MEDIUM_HALLUC_LOW    = 0.35   # yes_ratio in [0.35, 0.50) → MEDIUM hallucination
# yes_ratio < 0.35 → HIGH confidence hallucination

GDRIVE_ROOT        = "/content/drive/MyDrive/RelCheck_Data"
COCO_FRIENDLY_PATH = f"{GDRIVE_ROOT}/reltr_friendly_images.json"  # from COCO screening
NOCAPS_DIR         = f"{GDRIVE_ROOT}/nocaps_images"               # fallback
RBENCH_PATH        = f"{GDRIVE_ROOT}/rbench_qa.json"
RESULTS_DIR        = f"{GDRIVE_ROOT}/reltr_test_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# ── Together.ai LLM helper ────────────────────────────────────────────────────
def llm_call(prompt, model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
             max_tokens=512, temperature=0.0):
    """Call Together.ai API. Returns response text or None on failure."""
    headers = {"Authorization": f"Bearer {TOGETHER_API_KEY}",
               "Content-Type": "application/json"}
    payload = {"model": model,
               "messages": [{"role": "user", "content": prompt}],
               "max_tokens": max_tokens, "temperature": temperature}
    try:
        r = requests.post("https://api.together.xyz/v1/chat/completions",
                          headers=headers, json=payload, timeout=30)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"LLM call failed: {e}")
        return None

def vlm_call(image_b64, prompt, model="Qwen/Qwen3-VL-8B-Instruct", max_tokens=64):
    """Call Together.ai VLM (vision). Returns response text or None."""
    headers = {"Authorization": f"Bearer {TOGETHER_API_KEY}",
               "Content-Type": "application/json"}
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
            {"type": "text", "text": prompt}
        ]}],
        "max_tokens": max_tokens, "temperature": 0.0
    }
    try:
        r = requests.post("https://api.together.xyz/v1/chat/completions",
                          headers=headers, json=payload, timeout=30)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"VLM call failed: {e}")
        return None

print(f"Config: N_IMAGES={N_IMAGES}, SUPPORTED_THRESHOLD={SUPPORTED_THRESHOLD}")
print("Setup done.")


In [ ]:
# ── CELL 2: Load images ───────────────────────────────────────────────────────
import glob

# Prefer COCO images pre-screened for RelTR-friendly entity pairs
if os.path.exists(COCO_FRIENDLY_PATH):
    with open(COCO_FRIENDLY_PATH) as f:
        coco_data = json.load(f)
    all_imgs = [img['file_path'] for img in coco_data['images']
                if os.path.exists(img['file_path'])]
    img_source = 'COCO (RelTR-screened)'
    print(f"Loaded {len(all_imgs)} RelTR-friendly COCO images from screening")
    if len(all_imgs) == 0:
        print("  WARNING: 0 images on disk! Check COCO_IMG_DIR in screening notebook.")
else:
    all_imgs = sorted(glob.glob(f"{NOCAPS_DIR}/*.jpg") +
                      glob.glob(f"{NOCAPS_DIR}/*.png"))
    img_source = 'nocaps (fallback — run RelCheck_COCO_Screening.ipynb first!)'
    print(f"  WARNING: {COCO_FRIENDLY_PATH} not found — falling back to nocaps")
    print(f"  Found {len(all_imgs)} images in nocaps dir")

print(f"Image source: {img_source}")

random.seed(42)
selected_imgs = random.sample(all_imgs, min(N_IMAGES, len(all_imgs)))
print(f"Selected {len(selected_imgs)} images for this run (N_IMAGES={N_IMAGES})")

# ── Load R-Bench QA ───────────────────────────────────────────────────────────
if os.path.exists(RBENCH_PATH):
    with open(RBENCH_PATH) as f:
        rbench = json.load(f)
    rbench_by_img = {}
    for item in rbench:
        img_id = (item.get('image_id') or
                  os.path.splitext(os.path.basename(item.get('image','')))[0])
        rbench_by_img.setdefault(img_id, []).append(item)
    print(f"R-Bench: {len(rbench)} QA pairs across {len(rbench_by_img)} images")
else:
    rbench_by_img = {}
    print(f"  INFO: R-Bench QA file not found at {RBENCH_PATH}")
    print(f"  R-POPE evaluation will be skipped (Cell 7).")


In [ ]:
# ── CELL 3: Generate LLaVA captions (then unload model) ──────────────────────
# LLaVA-1.5-7B is ~14GB on GPU. We generate all captions then free GPU memory
# before loading GroundingDINO + RelTR.

LLAVA_CAPTIONS_PATH = f"{RESULTS_DIR}/llava_captions.json"

if os.path.exists(LLAVA_CAPTIONS_PATH):
    with open(LLAVA_CAPTIONS_PATH) as f:
        llava_captions = json.load(f)
    print(f"Loaded {len(llava_captions)} cached LLaVA captions")
else:
    !pip install -q transformers accelerate
    from transformers import LlavaProcessor, LlavaForConditionalGeneration
    import torchvision.transforms.functional as TF

    print("Loading LLaVA-1.5-7B...")
    llava_processor = LlavaProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
    llava_model = LlavaForConditionalGeneration.from_pretrained(
        "llava-hf/llava-1.5-7b-hf",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    llava_model.eval()
    print("LLaVA loaded ✅")

    llava_captions = {}
    LLAVA_PROMPT = "USER: <image>\nDescribe this image in detail, including all objects and their spatial relationships.\nASSISTANT:"

    for i, img_path in enumerate(selected_imgs):
        img_id = os.path.splitext(os.path.basename(img_path))[0]
        try:
            image = Image.open(img_path).convert("RGB")
            inputs = llava_processor(text=LLAVA_PROMPT, images=image,
                                     return_tensors="pt").to(llava_model.device)
            with torch.no_grad():
                out = llava_model.generate(**inputs, max_new_tokens=200,
                                           do_sample=False)
            caption = llava_processor.decode(out[0], skip_special_tokens=True)
            # Strip the prompt prefix
            if 'ASSISTANT:' in caption:
                caption = caption.split('ASSISTANT:')[-1].strip()
            llava_captions[img_id] = caption
            if (i+1) % 5 == 0:
                print(f"  {i+1}/{len(selected_imgs)}: {img_id} → {caption[:80]}...")
        except Exception as e:
            print(f"  Error on {img_id}: {e}")
            llava_captions[img_id] = ""

    with open(LLAVA_CAPTIONS_PATH, 'w') as f:
        json.dump(llava_captions, f, indent=2)
    print(f"\nGenerated + saved {len(llava_captions)} captions")

    # ── Free GPU memory before loading next models ────────────────────────────
    print("Unloading LLaVA...")
    del llava_model, llava_processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Preview a few captions
print("\nSample captions:")
for img_id, cap in list(llava_captions.items())[:3]:
    print(f"  [{img_id}] {cap[:120]}...")

In [ ]:
# ── CELL 4: Load GroundingDINO + RelTR ───────────────────────────────────────
# LLaVA is now unloaded. Load the visual verification models.

# ── GroundingDINO ─────────────────────────────────────────────────────────────
!pip install -q groundingdino-py 2>/dev/null || \
  pip install -q git+https://github.com/IDEA-Research/GroundingDINO.git 2>/dev/null

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

try:
    from groundingdino.util.inference import load_model as gdino_load, predict as gdino_predict
    from groundingdino.util import box_ops
    GDINO_CONFIG  = "/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
    GDINO_WEIGHTS = "/content/gdino_weights.pth"
    if not os.path.exists(GDINO_WEIGHTS):
        !wget -q -O /content/gdino_weights.pth \
          https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
    if not os.path.exists('/content/GroundingDINO'):
        !git clone -q https://github.com/IDEA-Research/GroundingDINO.git /content/GroundingDINO
        !pip install -q -e /content/GroundingDINO
    gdino_model = gdino_load(GDINO_CONFIG, GDINO_WEIGHTS)
    gdino_model.eval().to(device)
    GDINO_AVAILABLE = True
    print("GroundingDINO loaded ✅")
except Exception as e:
    print(f"GroundingDINO not available: {e}")
    GDINO_AVAILABLE = False

# ── RelTR ─────────────────────────────────────────────────────────────────────
if not os.path.exists('RelTR'):
    !git clone -q https://github.com/yrcong/RelTR.git
sys.path.insert(0, 'RelTR')

import argparse
from models import build_model as reltr_build_model
import torchvision.transforms as T

RELTR_CLASSES = [
    'N/A', 'airplane', 'animal', 'arm', 'bag', 'banana', 'basket', 'beach',
    'bear', 'bed', 'bench', 'bike', 'bird', 'board', 'boat', 'book', 'boot',
    'bottle', 'bowl', 'box', 'boy', 'branch', 'building', 'bus', 'cabinet',
    'cap', 'car', 'cat', 'chair', 'child', 'clock', 'coat', 'counter', 'cow',
    'cup', 'curtain', 'desk', 'dog', 'door', 'drawer', 'ear', 'elephant',
    'engine', 'eye', 'face', 'fence', 'finger', 'flag', 'flower', 'food',
    'fork', 'fruit', 'giraffe', 'girl', 'glass', 'glove', 'guy', 'hair',
    'hand', 'handle', 'hat', 'head', 'helmet', 'hill', 'horse', 'house',
    'jacket', 'jean', 'kid', 'kite', 'lady', 'lamp', 'laptop', 'leaf',
    'leg', 'letter', 'light', 'logo', 'man', 'men', 'motorcycle', 'mountain',
    'mouth', 'neck', 'nose', 'number', 'orange', 'pant', 'paper', 'paw',
    'people', 'person', 'phone', 'pillow', 'pizza', 'plane', 'plant', 'plate',
    'player', 'pole', 'post', 'pot', 'racket', 'railing', 'rock', 'roof',
    'room', 'screen', 'seat', 'sheep', 'shelf', 'shirt', 'shoe', 'short',
    'sidewalk', 'sign', 'sink', 'skateboard', 'ski', 'skier', 'sleeve',
    'snow', 'sock', 'stand', 'street', 'surfboard', 'table', 'tail', 'tie',
    'tile', 'tire', 'toilet', 'towel', 'tower', 'track', 'train', 'tree',
    'truck', 'trunk', 'umbrella', 'vase', 'vegetable', 'vehicle', 'wave',
    'wheel', 'window', 'windshield', 'wing', 'wire', 'woman', 'wood'
]
RELTR_REL_CLASSES = [
    'N/A', 'above', 'across', 'against', 'along', 'and', 'at', 'attached to',
    'behind', 'belonging to', 'between', 'carrying', 'covered in', 'covering',
    'eating', 'flying in', 'for', 'from', 'growing on', 'hanging from', 'has',
    'holding', 'in', 'in front of', 'laying on', 'looking at', 'lying on',
    'made of', 'mounted on', 'near', 'of', 'on', 'on back of', 'over',
    'painted on', 'parked on', 'part of', 'playing', 'riding', 'says',
    'sitting on', 'standing on', 'to', 'under', 'using', 'walking in',
    'walking on', 'watching', 'wearing', 'wears', 'with'
]
VG_CLASSES_SET = set(RELTR_CLASSES) - {'N/A'}

reltr_args = argparse.Namespace(
    backbone='resnet50', dilation=False, lr_backbone=0,
    return_interm_layers=False, position_embedding='sine',
    enc_layers=6, dec_layers=6, dim_feedforward=2048,
    hidden_dim=256, dropout=0.1, nheads=8,
    num_entities=100, num_triplets=200, pre_norm=False,
    set_cost_class=1, set_cost_bbox=5, set_cost_giou=2,
    set_iou_threshold=0.7, bbox_loss_coef=5, giou_loss_coef=2,
    rel_loss_coef=1, eos_coef=0.1, aux_loss=False,
    dataset='vg', device=device,
)
reltr_model, _, _ = reltr_build_model(reltr_args)

RELTR_CKPT = 'RelTR/ckpt/checkpoint0149.pth'
if not os.path.exists(RELTR_CKPT):
    raise FileNotFoundError(f"Upload RelTR checkpoint to {RELTR_CKPT}")
ckpt = torch.load(RELTR_CKPT, map_location=device, weights_only=False)
reltr_model.load_state_dict(ckpt['model'])
reltr_model.eval().to(device)
print(f"RelTR loaded ✅  ({os.path.getsize(RELTR_CKPT)/1e6:.0f} MB)")

reltr_transform = T.Compose([
    T.Resize(800), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
print(f"GPU used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── CELL 5: All helper functions ──────────────────────────────────────────────
import base64, io, random as _random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torchvision.transforms as T_gd

# ── Image utilities ───────────────────────────────────────────────────────────
def image_to_b64(image):
    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode()

def crop_image(image, bbox, pad=0.0):
    """Crop image to bbox [x1,y1,x2,y2] with optional extra padding fraction."""
    W, H = image.size
    x1,y1,x2,y2 = bbox
    if pad > 0:
        pw, ph = (x2-x1)*pad, (y2-y1)*pad
        x1,y1 = x1-pw, y1-ph
        x2,y2 = x2+pw, y2+ph
    return image.crop((max(0,x1), max(0,y1), min(W,x2), min(H,y2)))

def union_bbox(b1, b2, pad=0.15):
    """Union of two [x1,y1,x2,y2] bboxes with proportional padding."""
    x1 = min(b1[0], b2[0]); y1 = min(b1[1], b2[1])
    x2 = max(b1[2], b2[2]); y2 = max(b1[3], b2[3])
    pw, ph = (x2-x1)*pad, (y2-y1)*pad
    return [x1-pw, y1-ph, x2+pw, y2+ph]

# ── Triple extraction ─────────────────────────────────────────────────────────
EXTRACT_PROMPT = """Extract all (subject, relation, object) triples from this caption.
Caption: "{caption}"
Return JSON array only, no explanation:
[
  {{"subject": "...", "relation": "...", "object": "...", "type": "SPATIAL|ACTION|ATTRIBUTE"}}
]"""

def extract_triples(caption):
    """Extract typed triples from caption via Llama-3.3-70B."""
    resp = llm_call(EXTRACT_PROMPT.format(caption=caption), max_tokens=400)
    if not resp:
        return []
    try:
        m = re.search(r'\[.*\]', resp, re.DOTALL)
        return json.loads(m.group()) if m else []
    except:
        return []

# ── Entity-to-VG vocabulary mapping ──────────────────────────────────────────
ENTITY_TO_VG = {
    'man':'man','woman':'woman','boy':'boy','girl':'girl','person':'person',
    'people':'people','child':'child','kid':'kid','guy':'man','lady':'woman',
    'cat':'cat','dog':'dog','horse':'horse','cow':'cow','bird':'bird',
    'sheep':'sheep','elephant':'elephant','bear':'bear','giraffe':'giraffe',
    'bicycle':'bike','bike':'bike','motorbike':'motorcycle','motorcycle':'motorcycle',
    'aeroplane':'airplane','automobile':'car','vehicle':'vehicle','truck':'truck',
    'sofa':'chair','couch':'chair','desk':'desk','table':'table',
    'tv':'screen','television':'screen','monitor':'screen','laptop':'laptop',
    'phone':'phone','cellphone':'phone','smartphone':'phone',
    'jacket':'jacket','coat':'coat','jeans':'jean','trousers':'pant',
    'pants':'pant','shorts':'short','glasses':'glass','sunglasses':'glass',
    'cap':'cap','helmet':'helmet','hat':'hat','tie':'tie',
    'bottle':'bottle','bowl':'bowl','cup':'cup','bag':'bag','backpack':'bag',
    'umbrella':'umbrella','skateboard':'skateboard','surfboard':'surfboard',
    'sign':'sign','clock':'clock','book':'book','chair':'chair',
    'window':'window','door':'door','boat':'boat','train':'train',
    'bus':'bus','car':'car','kite':'kite','pizza':'pizza','tree':'tree',
    'flower':'flower','plant':'plant','suitcase':'bag','vase':'vase',
}
PREP_STOP = {'in','on','with','wearing','holding','at','of','near','by',
             'beside','behind','under','over','next','and','or'}
ADJ_SKIP  = {'a','an','the','large','small','big','little','old','young',
             'elderly','tall','short','long','red','blue','green','yellow',
             'black','white','brown','grey','gray','dark','bright','two',
             'three','several','some','wooden','metal','plastic','stone'}

def extract_head_noun(phrase):
    words = phrase.strip().lower().split()
    head = None
    for w in words:
        if w in PREP_STOP: break
        if w not in ADJ_SKIP: head = w
    return head

def entity_to_vg(entity):
    e = entity.strip().lower()
    if e in VG_CLASSES_SET: return e
    if e in ENTITY_TO_VG:   return ENTITY_TO_VG[e]
    h = extract_head_noun(e)
    if h:
        if h in VG_CLASSES_SET: return h
        if h in ENTITY_TO_VG:   return ENTITY_TO_VG[h]
    for w in e.split():
        if w in VG_CLASSES_SET: return w
        if w in ENTITY_TO_VG:   return ENTITY_TO_VG[w]
    return None

# ── GroundingDINO detection wrapper ──────────────────────────────────────────
GDINO_PREPROCESS = T_gd.Compose([
    T_gd.Resize(800),
    T_gd.ToTensor(),
    T_gd.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Broad query for KB construction (period-separated for GD multi-class mode)
KB_QUERY = ("person . man . woman . dog . cat . horse . bicycle . motorcycle . "
            "chair . couch . table . car . truck . bus . train . airplane . "
            "boat . bird . elephant . giraffe . bear . skateboard . surfboard . "
            "kite . umbrella . backpack . laptop . phone . book . clock . bottle")

def _gdino_detect(image, text_query, box_threshold=0.3, text_threshold=0.25):
    """
    Run GroundingDINO. Returns list of {'bbox':[x1,y1,x2,y2],'score':f,'label':s}.
    bbox is in pixel coordinates.
    """
    if not GDINO_AVAILABLE:
        return []
    try:
        W, H = image.size
        img_t = GDINO_PREPROCESS(image)
        boxes, logits, phrases = gdino_predict(
            gdino_model, img_t, text_query,
            box_threshold, text_threshold, device
        )
        results = []
        for box, logit, phrase in zip(boxes, logits, phrases):
            cx, cy, w, h = box.tolist()
            x1 = max(0.0, (cx - w/2) * W)
            y1 = max(0.0, (cy - h/2) * H)
            x2 = min(float(W), (cx + w/2) * W)
            y2 = min(float(H), (cy + h/2) * H)
            results.append({'bbox': [x1, y1, x2, y2],
                            'score': float(logit),
                            'label': phrase.lower().strip()})
        return results
    except Exception as e:
        print(f"  GD detect error: {e}")
        return []

# ── Visual Knowledge Base ─────────────────────────────────────────────────────
def build_visual_kb(image):
    """
    Construct visual KB for one image:
      'detections': {label → [{bbox, score}]}  from GroundingDINO (broad sweep)
      'description': VLM prose description (used in KB-first correction)
    """
    kb = {'detections': {}, 'description': ''}
    # Broad multi-class detection in a single GD forward pass
    for det in _gdino_detect(image, KB_QUERY):
        lbl = det['label']
        kb['detections'].setdefault(lbl, [])
        kb['detections'][lbl].append({'bbox': det['bbox'], 'score': det['score']})
    # VLM scene description
    desc = vlm_call(
        image_to_b64(image),
        "Describe the objects in this image and their spatial or action-based "
        "relationships. Be specific (e.g. 'a person is riding a horse', "
        "'a cat is sitting on a chair'). 3-4 sentences.",
        max_tokens=200
    )
    kb['description'] = desc or ''
    return kb

def get_entity_bbox(image, entity, kb=None):
    """
    Best-scoring bbox for entity.
    Checks KB detections first (fast), then runs a targeted GD query.
    """
    e = entity.strip().lower()
    vg = entity_to_vg(e) or e
    # Check KB detections
    if kb:
        for key, dets in kb['detections'].items():
            if dets and (key == e or key == vg or e in key or key in e
                         or vg in key or key in vg):
                return max(dets, key=lambda d: d['score'])['bbox']
    # Targeted GD query
    dets = _gdino_detect(image, e)
    if dets:
        return max(dets, key=lambda d: d['score'])['bbox']
    return None

# ── Relation type classification ──────────────────────────────────────────────
SPATIAL_KEYWORDS = {
    'on', 'under', 'above', 'below', 'on top of', 'beneath', 'atop',
    'in front of', 'behind', 'left of', 'right of', 'next to', 'beside',
    'near', 'between', 'inside', 'outside', 'over', 'across from',
    'in', 'at', 'against', 'along', 'adjacent to', 'in back of'
}
ACTION_KEYWORDS = {
    'riding', 'sitting on', 'standing on', 'lying on', 'holding', 'carrying',
    'wearing', 'eating', 'drinking', 'walking', 'running', 'jumping', 'flying',
    'watching', 'looking at', 'playing', 'using', 'driving', 'pulling',
    'pushing', 'throwing', 'catching', 'chasing', 'feeding', 'petting',
    'hugging', 'reading', 'typing', 'pointing', 'touching', 'grabbing',
    'climbing', 'herding', 'leading', 'following'
}

def classify_triple_type(relation, declared_type=None):
    """Returns 'SPATIAL', 'ACTION', or 'ATTRIBUTE'."""
    if declared_type in ('SPATIAL', 'ACTION', 'ATTRIBUTE'):
        return declared_type
    r = relation.lower().strip()
    if r in SPATIAL_KEYWORDS or any(kw in r for kw in SPATIAL_KEYWORDS):
        return 'SPATIAL'
    if r in ACTION_KEYWORDS or any(kw in r for kw in ACTION_KEYWORDS):
        return 'ACTION'
    return 'ATTRIBUTE'

# ── Spatial geometry verification (deterministic) ─────────────────────────────
def check_spatial_geometry(sub_bbox, obj_bbox, relation, dead_zone=0.08):
    """
    Rule-based geometry check. Returns True / False / None.
    None = ambiguous or unknown relation → caller should fallback to VQA.
    """
    sx1,sy1,sx2,sy2 = sub_bbox
    ox1,oy1,ox2,oy2 = obj_bbox
    sc_x,sc_y = (sx1+sx2)/2, (sy1+sy2)/2
    oc_x,oc_y = (ox1+ox2)/2, (oy1+oy2)/2
    avg_h = ((sy2-sy1)+(oy2-oy1))/2 + 1e-6
    avg_w = ((sx2-sx1)+(ox2-ox1))/2 + 1e-6

    r = relation.lower()
    if r in ('on', 'on top of', 'atop'):
        h_overlap = max(0, min(sx2,ox2) - max(sx1,ox1))
        return sc_y < oc_y - dead_zone*avg_h and h_overlap > 0.1*avg_w
    if r in ('under', 'below', 'beneath', 'underneath'):
        return sc_y > oc_y + dead_zone*avg_h
    if r in ('above', 'over'):
        return sc_y < oc_y - dead_zone*avg_h
    if r == 'in front of':
        return sc_y > oc_y + dead_zone*avg_h
    if r == 'behind':
        return sc_y < oc_y - dead_zone*avg_h
    if r in ('left of', 'to the left of'):
        return sc_x < oc_x - dead_zone*avg_w
    if r in ('right of', 'to the right of'):
        return sc_x > oc_x + dead_zone*avg_w
    if r in ('next to', 'beside', 'adjacent to', 'near', 'by'):
        dist = ((sc_x-oc_x)**2 + (sc_y-oc_y)**2)**0.5
        diag = (avg_w**2 + avg_h**2)**0.5
        return dist < diag * 2.5
    return None   # unknown spatial relation

# ── Crop-based contrastive VQA ────────────────────────────────────────────────
def crop_and_vqa_verify(image, subject, relation, obj, kb=None):
    """
    Verify (subject, relation, object) with 3 questions on a focused crop:
      - 2 standard yes/no questions
      - 1 contrastive forced-choice (A/B randomised to remove position bias)
    Returns (yes_ratio, confidence_level).
    confidence_level: 'SUPPORTED' | 'UNCERTAIN' | 'MEDIUM' | 'HIGH'
    """
    sub_bbox = get_entity_bbox(image, subject, kb)
    obj_bbox = get_entity_bbox(image, obj, kb)
    if sub_bbox and obj_bbox:
        crop = crop_image(image, union_bbox(sub_bbox, obj_bbox, pad=0.15))
    else:
        crop = image

    img_b64 = image_to_b64(crop)

    # 2 standard yes/no
    q1 = f"Is the {subject} {relation} the {obj}? Answer yes or no."
    q2 = f"Does the {subject} appear to be {relation} the {obj}? Answer yes or no."

    # 1 contrastive (randomise A/B to avoid position bias)
    flip = _random.random() > 0.5
    if not flip:
        cq = (f"Which describes this image better?\n"
              f"A: The {subject} is {relation} the {obj}.\n"
              f"B: The {subject} is NOT {relation} the {obj}.\n"
              f"Reply with A or B only.")
        c_yes = 'A'
    else:
        cq = (f"Which describes this image better?\n"
              f"A: The {subject} is NOT {relation} the {obj}.\n"
              f"B: The {subject} is {relation} the {obj}.\n"
              f"Reply with A or B only.")
        c_yes = 'B'

    yes_count = 0; total = 0
    for q in [q1, q2]:
        ans = vlm_call(img_b64, q, max_tokens=10)
        if ans:
            yes_count += int('yes' in ans.lower())
            total += 1
    c_ans = vlm_call(img_b64, cq, max_tokens=5)
    if c_ans:
        yes_count += int(c_ans.strip().upper().startswith(c_yes))
        total += 1

    yr = yes_count / total if total else 0.5

    if yr >= SUPPORTED_THRESHOLD:
        conf = 'SUPPORTED'
    elif yr >= UNCERTAIN_LOW:
        conf = 'UNCERTAIN'
    elif yr >= MEDIUM_HALLUC_LOW:
        conf = 'MEDIUM'
    else:
        conf = 'HIGH'
    return yr, conf

# ── Type-aware verification (core contribution) ────────────────────────────────
def type_aware_verify(image, subject, relation, obj, triple_type='ACTION', kb=None):
    """
    SPATIAL  → deterministic geometry → VQA fallback if geometry ambiguous
    ACTION / ATTRIBUTE → crop-based contrastive VQA
    Returns (yes_ratio, confidence_level, method_used)
    """
    if triple_type == 'SPATIAL':
        sub_bbox = get_entity_bbox(image, subject, kb)
        obj_bbox = get_entity_bbox(image, obj, kb)
        if sub_bbox and obj_bbox:
            geom = check_spatial_geometry(sub_bbox, obj_bbox, relation)
            if geom is not None:
                yr   = 1.0 if geom else 0.0
                conf = 'SUPPORTED' if geom else 'HIGH'
                return yr, conf, 'geometry'
        # Fall through: ambiguous geometry or detection failed
        yr, conf = crop_and_vqa_verify(image, subject, relation, obj, kb=kb)
        return yr, conf, 'geometry_fallback_vqa'

    yr, conf = crop_and_vqa_verify(image, subject, relation, obj, kb=kb)
    return yr, conf, 'crop_vqa'

# ── RelTR scene graph builder ─────────────────────────────────────────────────
def build_reltr_scene_graph(image, conf_threshold=CONF_THRESHOLD):
    """Run RelTR on PIL image, return list of triple dicts sorted by confidence."""
    img_t = reltr_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = reltr_model(img_t)

    probas     = outputs['rel_logits'].softmax(-1)[0, :, :-1]
    probas_sub = outputs['sub_logits'].softmax(-1)[0, :, :-1]
    probas_obj = outputs['obj_logits'].softmax(-1)[0, :, :-1]

    keep = torch.nonzero(
        (probas.max(-1)[0] > conf_threshold) &
        (probas_sub.max(-1)[0] > conf_threshold) &
        (probas_obj.max(-1)[0] > conf_threshold)
    ).flatten()

    bboxes_sub_raw = outputs['sub_boxes'][0]
    bboxes_obj_raw = outputs['obj_boxes'][0]
    W, H = image.size

    scene_graph = []
    for idx in keep:
        # Embed bboxes in each entry (safe to sort after)
        def _to_px(b):
            cx,cy,w,h = b.tolist()
            return [max(0,(cx-w/2)*W), max(0,(cy-h/2)*H),
                    min(W,(cx+w/2)*W), min(H,(cy+h/2)*H)]
        scene_graph.append({
            'subject':        RELTR_CLASSES[probas_sub[idx].argmax()],
            'predicate':      RELTR_REL_CLASSES[probas[idx].argmax()],
            'object':         RELTR_CLASSES[probas_obj[idx].argmax()],
            'predicate_conf': float(probas[idx].max()),
            'bbox_sub':       _to_px(bboxes_sub_raw[idx]),
            'bbox_obj':       _to_px(bboxes_obj_raw[idx]),
        })
    scene_graph.sort(key=lambda x: -x['predicate_conf'])
    return scene_graph

# ── RelTR lookup ──────────────────────────────────────────────────────────────
def reltr_lookup(scene_graph, subj, obj):
    """Find best-confidence RelTR triple matching (subj, ?, obj) by VG synonyms."""
    vg_s, vg_o = entity_to_vg(subj), entity_to_vg(obj)
    if vg_s is None or vg_o is None:
        return None, 'oov'

    matches = []
    for t in scene_graph:
        ts = entity_to_vg(t['subject'])  or t['subject']
        to = entity_to_vg(t['object'])   or t['object']
        fwd = ((ts==vg_s or vg_s in ts or ts in vg_s) and
               (to==vg_o or vg_o in to or to in vg_o))
        rev = ((ts==vg_o or vg_o in ts or ts in vg_o) and
               (to==vg_s or vg_s in to or to in vg_s))
        if fwd:
            matches.append(t)
        elif rev:
            rt = dict(t); rt['subject'], rt['object'] = t['object'], t['subject']
            matches.append(rt)

    if not matches:
        return None, 'no_match'
    return max(matches, key=lambda x: x['predicate_conf']), 'ok'

# ── Coherency check ───────────────────────────────────────────────────────────
COHERENCY_PROMPT = """You are evaluating a caption correction.

Original caption: "{original}"
Corrected caption: "{corrected}"
Change: replaced "{old_rel}" with "{new_rel}" between "{subj}" and "{obj}"

Answer YES if ALL hold:
1. Corrected caption is grammatically correct
2. "{new_rel}" makes semantic sense between "{subj}" and "{obj}"
3. Caption is not garbled or repetitive

Answer NO if any fails.
Reply: YES or NO, then one-line reason."""

def check_coherency(original, corrected, subj, old_rel, new_rel, obj):
    """Returns (accept: bool, reason: str)."""
    if original.strip() == corrected.strip():
        return False, 'caption unchanged'
    resp = llm_call(COHERENCY_PROMPT.format(
        original=original, corrected=corrected,
        old_rel=old_rel, new_rel=new_rel, subj=subj, obj=obj
    ), max_tokens=80)
    if not resp:
        return False, 'LLM unavailable'
    return resp.strip().upper().startswith('YES'), resp.strip()

def apply_edit(caption, old_rel, new_rel):
    """Replace first occurrence of old_rel with new_rel (case-insensitive)."""
    return re.sub(re.escape(old_rel), new_rel, caption, count=1, flags=re.IGNORECASE)

# ── KB-first correction cascade ────────────────────────────────────────────────
CORRECTION_CANDIDATES = [
    'riding', 'sitting on', 'standing near', 'holding',
    'walking with', 'next to', 'lying on', 'wearing',
    'carrying', 'on'
]

def _quick_verify(image, subj, new_rel, obj):
    """Single yes/no VQA to confirm a correction before accepting."""
    ans = vlm_call(image_to_b64(image),
                   f"Is the {subj} {new_rel} the {obj}? Answer yes or no.",
                   max_tokens=10)
    return bool(ans and 'yes' in ans.lower())

def kb_first_correct(image, kb, scene_graph, subj, old_rel, obj):
    """
    4-stage correction cascade (decreasing hallucination risk):
      Stage 0: RelTR scene graph  — visual classifier, no text generation
      Stage 1: KB description     — already-grounded VLM prose
      Stage 2: Constrained VLM   — forced choice from 10 candidates
      Stage 3: Open VLM          — last resort, short generation
    Returns (new_relation, source, accepted).
    """
    # Stage 0 — RelTR ──────────────────────────────────────────────────────────
    best, status = reltr_lookup(scene_graph, subj, obj)
    if status == 'ok' and best:
        nr = best['predicate']
        TRIVIAL = {'N/A', 'near', 'and', 'of', 'with', 'has', 'part of'}
        if nr not in TRIVIAL and nr.lower() != old_rel.lower():
            if _quick_verify(image, subj, nr, obj):
                return nr, 'reltr', True

    # Stage 1 — KB description ────────────────────────────────────────────────
    desc = kb.get('description', '')
    if desc and subj.lower() in desc.lower() and obj.lower() in desc.lower():
        kb_prompt = (
            f'Description: "{desc}"\n'
            f'What is the relationship between "{subj}" and "{obj}" '
            f'in the description? Give ONLY a 2-4 word relation phrase '
            f'(e.g. "sitting on"). If unclear, reply UNKNOWN.'
        )
        kb_rel = llm_call(kb_prompt, max_tokens=15)
        if kb_rel and kb_rel.strip().upper() != 'UNKNOWN':
            nr = kb_rel.strip().lower().rstrip('.')
            if nr and nr != old_rel.lower():
                if _quick_verify(image, subj, nr, obj):
                    return nr, 'kb_description', True

    # Stage 2 — Constrained VLM selection ─────────────────────────────────────
    opts = '\n'.join(f'{chr(65+i)}. {c}' for i,c in enumerate(CORRECTION_CANDIDATES))
    img_b64 = image_to_b64(image)
    sel = vlm_call(img_b64,
                   f'Caption says "{subj} {old_rel} {obj}" — relation may be wrong.\n'
                   f'Which relation best fits what you see?\n{opts}\n'
                   f'Reply with ONLY the letter (A–{chr(64+len(CORRECTION_CANDIDATES))}).',
                   max_tokens=5)
    if sel:
        idx = ord(sel.strip()[0].upper()) - 65
        if 0 <= idx < len(CORRECTION_CANDIDATES):
            nr = CORRECTION_CANDIDATES[idx]
            if nr.lower() != old_rel.lower():
                if _quick_verify(image, subj, nr, obj):
                    return nr, 'vlm_constrained', True

    # Stage 3 — Open VLM ───────────────────────────────────────────────────────
    open_r = vlm_call(img_b64,
                      f'Caption: "{subj} {old_rel} {obj}" — relation is wrong.\n'
                      f'What is the correct relation? Return ONLY 2-4 words.',
                      max_tokens=20)
    if open_r:
        nr = open_r.strip().lower().rstrip('.')
        if nr and nr != old_rel.lower():
            if _quick_verify(image, subj, nr, obj):
                return nr, 'vlm_open', True

    return None, 'failed', False

print('All helper functions defined. ✅')
print(f'  Geometry rules: {len(SPATIAL_KEYWORDS)} spatial keywords')
print(f'  Action keywords: {len(ACTION_KEYWORDS)}')
print(f'  Correction candidates: {len(CORRECTION_CANDIDATES)}')


In [ ]:
# ── CELL 6: Full pipeline per image ──────────────────────────────────────────
# For each image:
#   1. Build Visual KB  (GroundingDINO + VLM description)
#   2. Build RelTR scene graph
#   3. Extract triples from LLaVA caption
#   4. Type-aware verification (SPATIAL→geometry, ACTION→crop VQA)
#   5. Confidence gate: HIGH + MEDIUM → correct; UNCERTAIN → skip; SUPPORTED → keep
#   6. KB-first correction cascade: RelTR → KB desc → constrained VLM → open VLM
#   7. Coherency check before applying correction

PIPELINE_RESULTS_PATH = f"{RESULTS_DIR}/pipeline_results.json"

if os.path.exists(PIPELINE_RESULTS_PATH):
    with open(PIPELINE_RESULTS_PATH) as f:
        pipeline_results = json.load(f)
    print(f"Loaded {len(pipeline_results)} cached results from disk")
else:
    pipeline_results = {}

# ── Stats tracking ────────────────────────────────────────────────────────────
stats = {
    # Verification counts
    'total_triples': 0,
    'supported': 0, 'uncertain_skip': 0,
    'halluc_medium': 0, 'halluc_high': 0,
    # Routing method
    'method_geometry': 0, 'method_geo_vqa': 0, 'method_crop_vqa': 0,
    # Correction source
    'src_reltr': 0, 'src_kb': 0, 'src_vlm_const': 0, 'src_vlm_open': 0,
    'cascade_failed': 0,
    # Coherency gate
    'coherency_accept': 0, 'coherency_reject': 0,
    # Images
    'images_corrected': 0,
}

# ── Per-image pipeline ────────────────────────────────────────────────────────
for img_path in selected_imgs:
    img_id = os.path.splitext(os.path.basename(img_path))[0]

    if img_id in pipeline_results:
        continue   # already cached

    caption = llava_captions.get(img_id, '')
    if not caption:
        print(f"  [{img_id}] no caption — skipping")
        continue

    print(f"\n── {img_id} ──────────────────────────")
    print(f"  Caption ({len(caption.split())} words): {caption[:100]}...")

    image = Image.open(img_path).convert('RGB')

    # ── 1. Visual KB ─────────────────────────────────────────────────────────
    print(f"  Building KB...", end=' ', flush=True)
    kb = build_visual_kb(image)
    n_det = sum(len(v) for v in kb['detections'].values())
    print(f"{len(kb['detections'])} entity types, {n_det} detections | "
          f"desc={len(kb['description'])} chars")

    # ── 2. RelTR scene graph ──────────────────────────────────────────────────
    scene_graph = build_reltr_scene_graph(image)
    non_trivial_sg = [t for t in scene_graph
                      if t['predicate'] not in {'N/A','near','and','of','with','has'}]
    print(f"  RelTR: {len(scene_graph)} triples, "
          f"{len(non_trivial_sg)} non-trivial")

    # ── 3. Extract triples from caption ──────────────────────────────────────
    triples = extract_triples(caption)
    print(f"  Triples: {len(triples)}")
    stats['total_triples'] += len(triples)

    corrected_caption = caption
    triple_results    = []
    image_corrected   = False

    for t in triples:
        subj      = t.get('subject','').strip()
        rel       = t.get('relation','').strip()
        obj       = t.get('object','').strip()
        decl_type = t.get('type','').strip()
        if not subj or not rel or not obj:
            continue

        triple_type = classify_triple_type(rel, decl_type)

        # ── 4. Type-aware verification ────────────────────────────────────────
        yes_ratio, confidence, method = type_aware_verify(
            image, subj, rel, obj, triple_type=triple_type, kb=kb
        )

        # Track routing method
        if method == 'geometry':          stats['method_geometry'] += 1
        elif method == 'geometry_fallback_vqa': stats['method_geo_vqa'] += 1
        else:                             stats['method_crop_vqa'] += 1

        tr = {
            'subject': subj, 'relation': rel, 'object': obj,
            'type': triple_type, 'yes_ratio': round(yes_ratio, 3),
            'confidence': confidence, 'method': method,
            'correction': None, 'correction_source': None,
        }

        # ── 5. Confidence gate ────────────────────────────────────────────────
        if confidence == 'SUPPORTED':
            stats['supported'] += 1
            triple_results.append(tr)
            continue

        if confidence == 'UNCERTAIN':
            stats['uncertain_skip'] += 1
            triple_results.append(tr)
            continue

        # HIGH or MEDIUM → attempt correction
        if confidence == 'HIGH':
            stats['halluc_high'] += 1
        else:
            stats['halluc_medium'] += 1

        print(f"  ⚠ {confidence} ({triple_type}): ({subj}, {rel}, {obj}) "
              f"yr={yes_ratio:.2f} [{method}]")

        # ── 6. KB-first correction cascade ───────────────────────────────────
        new_rel, source, accepted = kb_first_correct(
            image, kb, scene_graph, subj, rel, obj
        )

        if not accepted:
            stats['cascade_failed'] += 1
            print(f"    Cascade failed — keeping original")
            triple_results.append(tr)
            continue

        # Track correction source
        if   source == 'reltr':         stats['src_reltr'] += 1
        elif source == 'kb_description':stats['src_kb'] += 1
        elif source == 'vlm_constrained':stats['src_vlm_const'] += 1
        else:                           stats['src_vlm_open'] += 1

        print(f"    [{source}] '{rel}' → '{new_rel}'")

        # ── 7. Coherency check ────────────────────────────────────────────────
        candidate = apply_edit(corrected_caption, rel, new_rel)
        accept, reason = check_coherency(
            original=corrected_caption, corrected=candidate,
            subj=subj, old_rel=rel, new_rel=new_rel, obj=obj
        )
        flag = '✅' if accept else '❌'
        print(f"    Coherency: {flag} — {reason[:80]}")

        if accept:
            stats['coherency_accept'] += 1
            corrected_caption = candidate
            image_corrected   = True
            tr['correction']        = new_rel
            tr['correction_source'] = source
        else:
            stats['coherency_reject'] += 1

        triple_results.append(tr)

    if image_corrected:
        stats['images_corrected'] += 1

    pipeline_results[img_id] = {
        'original_caption':  caption,
        'corrected_caption': corrected_caption,
        'triples':           triple_results,
        'scene_graph_size':  len(scene_graph),
        'kb_entities':       len(kb['detections']),
        'kb_description':    kb['description'][:200],
        'corrected':         image_corrected,
    }

    # Checkpoint every 5 images
    if len(pipeline_results) % 5 == 0:
        with open(PIPELINE_RESULTS_PATH, 'w') as f:
            json.dump(pipeline_results, f, indent=2)
        print(f"  ✓ checkpoint saved ({len(pipeline_results)} images)")

# Final save
with open(PIPELINE_RESULTS_PATH, 'w') as f:
    json.dump(pipeline_results, f, indent=2)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═'*56}")
print(f"Pipeline complete: {len(pipeline_results)} images processed")

total_h = stats['halluc_high'] + stats['halluc_medium']
print(f"\n── Verification ──────────────────────────────────────")
print(f"  Total triples       : {stats['total_triples']}")
print(f"  Supported           : {stats['supported']}")
print(f"  Uncertain (skipped) : {stats['uncertain_skip']}")
print(f"  MEDIUM hallucinated : {stats['halluc_medium']}")
print(f"  HIGH hallucinated   : {stats['halluc_high']}")
print(f"  Total to correct    : {total_h}")

print(f"\n── Routing (type-aware) ──────────────────────────────")
print(f"  Geometry (SPATIAL)  : {stats['method_geometry']}")
print(f"  Geo→VQA fallback    : {stats['method_geo_vqa']}")
print(f"  Crop VQA (ACT/ATTR) : {stats['method_crop_vqa']}")

print(f"\n── Correction cascade ────────────────────────────────")
print(f"  Stage 0 RelTR       : {stats['src_reltr']}")
print(f"  Stage 1 KB desc     : {stats['src_kb']}")
print(f"  Stage 2 constrained : {stats['src_vlm_const']}")
print(f"  Stage 3 open VLM    : {stats['src_vlm_open']}")
print(f"  Cascade failed      : {stats['cascade_failed']}")
print(f"  Coherency accept    : {stats['coherency_accept']}")
print(f"  Coherency reject    : {stats['coherency_reject']}")
print(f"  Images corrected    : {stats['images_corrected']} / "
      f"{len(pipeline_results)}")


In [ ]:
# ── CELL 7: R-POPE LLM-Judge evaluation ──────────────────────────────────────
# For each image with R-Bench QA pairs, ask Llama to answer using
# the original vs. corrected caption. Compare accuracy.

RPOPE_PROMPT = """Based ONLY on the following image description, answer the question.
Do NOT use any external knowledge. If the description doesn't mention it, answer 'no'.

Description: "{caption}"
Question: {question}

Answer with just 'yes' or 'no'."""

orig_correct = 0
corr_correct = 0
total_q = 0

for img_id, result in pipeline_results.items():
    qas = rbench_by_img.get(img_id, [])
    if not qas:
        continue

    orig_cap = result['original_caption']
    corr_cap = result['corrected_caption']

    for qa in qas:
        question = qa.get('question', qa.get('q', ''))
        gt       = str(qa.get('answer', qa.get('a', ''))).lower().strip()
        if not question or gt not in ('yes','no'):
            continue

        total_q += 1

        # Original caption
        ans_orig = llm_call(RPOPE_PROMPT.format(caption=orig_cap, question=question),
                            max_tokens=5)
        orig_correct += int(bool(ans_orig) and ans_orig.lower().startswith(gt))

        # Corrected caption
        ans_corr = llm_call(RPOPE_PROMPT.format(caption=corr_cap, question=question),
                            max_tokens=5)
        corr_correct += int(bool(ans_corr) and ans_corr.lower().startswith(gt))

print(f"\n{'═'*50}")
print(f"R-POPE LLM-Judge ({total_q} questions)")
print(f"  Original accuracy:  {orig_correct}/{total_q} = {orig_correct/max(total_q,1)*100:.1f}%")
print(f"  Corrected accuracy: {corr_correct}/{total_q} = {corr_correct/max(total_q,1)*100:.1f}%")
print(f"  Delta:              {(corr_correct-orig_correct)/max(total_q,1)*100:+.1f}%")

In [ ]:
# ── CELL 8: Coverage + qualitative analysis ───────────────────────────────────
import matplotlib.pyplot as plt
from collections import Counter

# RelTR coverage breakdown
total_hal = stats['hallucinated']
if total_hal > 0:
    print("RelTR Coverage (of hallucinated triples):")
    print(f"  OOV (skipped instantly):  {stats['reltr_oov']:>3}  "
          f"({stats['reltr_oov']/total_hal*100:.0f}%)")
    print(f"  In-vocab, no match:       {stats['reltr_no_match']:>3}  "
          f"({stats['reltr_no_match']/total_hal*100:.0f}%)")
    print(f"  RelTR fired:              {stats['reltr_fired']:>3}  "
          f"({stats['reltr_fired']/total_hal*100:.0f}%)")
    print(f"  → Coherency accepted:     {stats['coherency_accept']:>3}  "
          f"({stats['coherency_accept']/max(stats['reltr_fired'],1)*100:.0f}% of fired)")
    print(f"  → Coherency rejected:     {stats['coherency_reject']:>3}")
    print(f"  VLM fallback total:       {stats['vlm_fallback']:>3}")

# Pie chart
if total_hal > 0:
    labels = ['OOV (skip)', 'In-vocab\nno match', 'RelTR\naccepted', 'RelTR\nrejected→VLM']
    sizes  = [stats['reltr_oov'], stats['reltr_no_match'],
               stats['coherency_accept'], stats['coherency_reject']]
    colors = ['#d3d3d3','#ffcc99','#90ee90','#ffb3b3']
    fig, ax = plt.subplots(figsize=(7,5))
    ax.pie([s for s in sizes if s > 0],
           labels=[l for l,s in zip(labels,sizes) if s > 0],
           colors=[c for c,s in zip(colors,sizes) if s > 0],
           autopct='%1.0f%%', startangle=90)
    ax.set_title('RelTR Coverage — Hallucinated Triple Routing', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/reltr_coverage_pie.png', dpi=150)
    plt.show()

# Qualitative examples
print("\nQualitative corrections (RelTR accepted):")
n_shown = 0
for img_id, result in pipeline_results.items():
    if not result['corrected']: continue
    for t in result['triples']:
        if t.get('source') == 'reltr' and t.get('correction'):
            print(f"  [{img_id}]")
            print(f"    BEFORE: {result['original_caption'][:100]}")
            print(f"    AFTER:  {result['corrected_caption'][:100]}")
            print(f"    CHANGE: ({t['subject']}, {t['relation']}, {t['object']}) "
                  f"→ '{t['correction']}' (conf={t.get('reltr_conf',0):.2f})")
            print()
            n_shown += 1
            if n_shown >= 5: break
    if n_shown >= 5: break

print(f"Results saved to: {RESULTS_DIR}")